In [1]:
from stable_baselines3 import PPO
import numpy as np
import pandas as pd
from typing import Any
from OPSDesignerEnv import OPSDesignerEnv

C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


C:\Users\noton\miniconda3\envs\OPSdesign2\Lib\site-packages\deepchem\feat\smiles_tokenizer.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
fragment_pool = pd.read_csv("source/SMILES_list_100.csv")["SMILES"].tolist()
targetOPS = [-5.9155, -2.7331, 2.5561, 0.0639, 2.4273, 0.1288]

eval_env = OPSDesignerEnv(smiles=fragment_pool,
                          target_ops=targetOPS,
                          min_k=1,
                          max_k=3,
                          max_steps=4,
                          eps=1e-8,
                          r_cap=100,
                         )

def generate_SMILES(model, env, n_episodes=200, base_seed=42, deterministic=False):
    rows = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=base_seed + ep)
        terminated = truncated = False
        total_reward = 0.0
        last_info = {}

        while not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, terminated, truncated, info = env.step(int(action))
            total_reward += float(r)
            last_info = info

        best = last_info.get("best_smiles")
        selected = last_info.get("selected_smiles")
        smiles = best if best is not None else selected

        if isinstance(smiles, list):
            smiles = tuple(smiles)
        if isinstance(selected, list):
            selected = tuple(selected)

        rows.append({"episode": ep, "smiles": smiles, "selected_fragments": selected, "reward": total_reward})
        #print(f"episode {ep} done.")
        
    return pd.DataFrame(rows)

In [3]:
model = PPO.load("RLmodels/ppo_rm_TL.zip", device="cpu")
df = generate_SMILES(model, eval_env, n_episodes=50)
df.to_csv("generated_OPSs/rl_candidates_TL.csv", index=False)
df

,episode,smiles,selected_fragments,reward
0,0,O=C1NC(=O)c2ccc(N3C(=O)c4cccc5c(-n6c7ccccc7c7c...,"(O=C1NC(=O)c2cccc3cccc1c23, c1ccc2c(c1)Nc1cccc...",18.800536
1,1,O=C1c2cccc3c(N4C(=O)c5cccccccc(c5)C4=O)ccc(c23...,"(O=C1NC(=O)c2cccc3cccc1c23, O=C1NC(=O)c2cccc3c...",10.710855
2,2,O=C1c2cccc3c(-n4c5ccccc5c5ccccc54)ccc(c23)C(=O...,"(O=C1NC(=O)c2cccc3cccc1c23, c1ccc2c(c1)Nc1cccc...",15.896087
3,3,O=C1c2cccc3c(-n4c5ccccc5c5ccccc54)ccc(c23)C(=O...,"(O=C1NC(=O)c2cccc3cccc1c23, c1ccc2c(c1)Nc1cccc...",15.896087
4,4,O=C1c2cccc3c(-n4c5ccccc5c5ccccc54)ccc(c23)C(=O...,"(O=C1NC(=O)c2cccc3cccc1c23, c1ccc2c(c1)Nc1cccc...",15.896087
5,5,Cc1cc(C#N)cc(C)c1-n1c2ccccc2c2cc(-c3cc4ccccc4n...,"(Cn1ccc2ccccc21, c1ccc2c(c1)Nc1ccccc12, Cc1cc(...",3.891983
6,6,O=C1c2cccc3c(-n4c5ccccc5c5ccccc54)ccc(c23)C(=O...,"(O=C1NC(=O)c2cccc3cccc1c23, c1ccc2c(c1)Nc1cccc...",15.896087
7,7,Cn1cc(N2C(=O)c3cccc4c(-n5c6ccccc6c6ccccc65)ccc...,"(O=C1NC(=O)c2cccc3cccc1c23, c1ccc2c(c1)Nc1cccc...",13.255830
8,8,O=C1c2cccc3c(N4C(=O)c5ccccc5C4=O)ccc(c23)C(=O)...,"(O=C1NC(=O)c2cccc3cccc1c23, O=C1NC(=O)c2ccccc2...",10.688084
9,9,N#Cc1cc(C#N)c(-n2c3ccccc3c3ccccc32)cc1-n1c2ccc...,"(N#Cc1cccc(C#N)c1, c1ccc2c(c1)Nc1ccccc12, c1cc...",21.257013
